# 03 — XGBoost on TF-IDF

Same vectorizer family as the baseline, non-linear classifier. Feature-importance and SHAP are computed in `07_explainability.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib; sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src import config as C
from src.data_utils import get_split
from src.evaluate import evaluate_and_log
from src.models import xgboost_model

In [ ]:
for lang in C.LANGUAGES:
    print(f'\n=== XGBoost :: {lang.upper()} ===')
    try:
        tr, va, te = get_split(lang)
    except FileNotFoundError as e:
        print(f'  skipped: {e}'); continue
    pipe = xgboost_model.train(tr, va, lang)
    y_pred = xgboost_model.predict(pipe, te, lang)
    metrics = evaluate_and_log(te['label'].values, y_pred, model_name='xgboost', lang=lang)
    print('  metrics:', {k: round(v, 4) for k, v in metrics.items()})
    xgboost_model.save(pipe, C.RESULTS_DIR / 'checkpoints' / f'xgboost_{lang}.joblib')